In [0]:
%run /Workspace/Users/shreyash270204@outlook.com/databricks/utilities/config.py

Config loaded. STORAGE_ACCOUNT=financestorage1 SNAPSHOT_DATE=latest per exchange TAXONOMY_VERSION=v1


In [0]:
# Imports
from pyspark.sql import functions as F, types as T
from delta.tables import DeltaTable

In [0]:
# Exchange Dimension Data
dim_exchange_data = [

    ("BER", "Berlin Stock Exchange",           "Germany",        "DE", "DEU", "Europe",        "Equity"),

    ("BSE", "Bombay Stock Exchange",            "India",          "IN", "IND", "Asia",          "Equity"),

    ("FRA", "Frankfurt Stock Exchange",         "Germany",        "DE", "DEU", "Europe",        "Equity"),

    ("GER", "XETRA",                            "Germany",        "DE", "DEU", "Europe",        "Equity"),

    ("JPX", "Japan Exchange Group",             "Japan",          "JP", "JPN", "Asia",          "Equity"),

    ("LSE", "London Stock Exchange",            "United Kingdom", "GB", "GBR", "Europe",        "Equity"),

    ("NSE", "National Stock Exchange of India", "India",          "IN", "IND", "Asia",          "Equity"),

    ("SHZ", "Shenzhen Stock Exchange",          "China",          "CN", "CHN", "Asia",          "Equity"),

    ("VIE", "Vienna Stock Exchange",            "Austria",        "AT", "AUT", "Europe",        "Equity"),

    ("NYQ", "New York Stock Exchange",          "United States",  "US", "USA", "North America", "Equity"),

]

In [0]:
# Exchange Schema
dim_exchange_schema = T.StructType([

    T.StructField("exchange_code", T.StringType(), False),

    T.StructField("exchange_name", T.StringType(), False),

    T.StructField("country_name", T.StringType(), False),

    T.StructField("iso2", T.StringType(), False),

    T.StructField("iso3", T.StringType(), False),

    T.StructField("region", T.StringType(), False),

    T.StructField("market_type", T.StringType(), False),

])

In [0]:
# Create Exchange DataFrame
dim_exchange_df = (

    spark.createDataFrame(dim_exchange_data, schema=dim_exchange_schema)

    .withColumn("exchange_key", F.sha2(F.col("exchange_code"), 256))

    .withColumn("taxonomy_version", F.lit(TAXONOMY_VERSION))

    .withColumn("is_current", F.lit(True))

)

In [0]:
# Write dim_exchange
(

    dim_exchange_df.write

    .format("delta")

    .mode("overwrite")

    .option("overwriteSchema", "true")

    .save(silver_path("dim_exchange"))

)

print(f"dim_exchange written: {dim_exchange_df.count()} rows (expected 10)")

dim_exchange written: 10 rows (expected 10)


In [0]:
country_frames = []
for exch, classes in EXCHANGE_ASSET_COVERAGE.items():
    if "equities" not in classes:
        continue
    path = latest_snapshot_path("equities", exch)
    if path is None:
        continue
    df = (
        spark.read.option("header", True).option("inferSchema", True)
        .option("multiLine", True).option("escape", "\"")
        .csv(f"{path}/*.csv")
        .select("country")
        .where(F.col("country").isNotNull())
    )
    country_frames.append(df)

discovered_countries = country_frames[0]
for f in country_frames[1:]:
    discovered_countries = discovered_countries.unionByName(f)
discovered_countries = discovered_countries.distinct().withColumnRenamed("country", "country_name")

exchange_derived_countries = dim_exchange_df.select("country_name", "iso2", "iso3", "region").distinct()

dim_country_df = (
    discovered_countries
    .join(exchange_derived_countries, on="country_name", how="left")
    .withColumn("country_key", F.sha2(F.col("country_name"), 256))
    .withColumn("sub_region", F.lit(None).cast("string"))
    .withColumn("economic_region", F.lit(None).cast("string"))
    .withColumn("needs_review", F.col("iso2").isNull())
)

(
    dim_country_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(silver_path("dim_country"))
)

print(f"dim_country written: {dim_country_df.count()} rows")
print(f"  of which needs_review (no curated iso2/region yet): {dim_country_df.where('needs_review = true').count()}")

dim_country written: 95 rows
  of which needs_review (no curated iso2/region yet): 88


In [0]:
# Write dim_country
(
    dim_country_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(silver_path("dim_country"))
)
print(f"dim_country written: {dim_country_df.count()} rows")

dim_country written: 95 rows


In [0]:
# Currency Lookup

CURRENCY_NAME_LOOKUP = {
    "USD": ("US Dollar", "North America", "Fiat"),
    "EUR": ("Euro", "Europe", "Fiat"),
    "GBP": ("British Pound", "Europe", "Fiat"),
    "GBp": ("British Pound (pence)", "Europe", "Fiat"),
    "JPY": ("Japanese Yen", "Asia", "Fiat"),
    "INR": ("Indian Rupee", "Asia", "Fiat"),
    "CNY": ("Chinese Yuan", "Asia", "Fiat"),
    "HKD": ("Hong Kong Dollar", "Asia", "Fiat"),
    "CHF": ("Swiss Franc", "Europe", "Fiat"),
    "AUD": ("Australian Dollar", "Oceania", "Fiat"),
    "CAD": ("Canadian Dollar", "North America", "Fiat"),
}

In [0]:
def read_all_bronze(asset_class: str):
    frames = []
    for exch, classes in EXCHANGE_ASSET_COVERAGE.items():
        if asset_class not in classes:
            continue
        path = latest_snapshot_path(asset_class, exch)
        if path is None:
            continue
        df = (
            spark.read.option("header", True).option("inferSchema", True)
            .option("multiLine", True).option("escape", "\"")
            .csv(f"{path}/*.csv")
            .withColumn("_source_exchange", F.lit(exch))
            .withColumn("_snapshot_date", F.lit(path.split("snapshot_date=")[-1]))
        )
        frames.append(df)
    if not frames:
        return None
    out = frames[0]
    for f in frames[1:]:
        out = out.unionByName(f, allowMissingColumns=True)
    return out

In [0]:
# Find Distinct Currencies
distinct_currencies = None

for ac in ASSET_CLASSES:

    df = read_all_bronze(ac)

    if df is None:

        continue

    cur = df.select("currency").where(F.col("currency").isNotNull()).distinct()

    distinct_currencies = cur if distinct_currencies is None else distinct_currencies.unionByName(cur).distinct()

In [0]:
# Create Currency Lookup DataFrame
lookup_rows = [(code, name, region, ctype) for code, (name, region, ctype) in CURRENCY_NAME_LOOKUP.items()]

lookup_df = spark.createDataFrame(
    lookup_rows,
    ["currency", "currency_name", "currency_region", "currency_type"]
)

In [0]:
# Build Incoming Currency Dimension
dim_currency_incoming = (

    distinct_currencies

    .join(lookup_df, on="currency", how="left")

    .withColumn("currency_key", F.sha2(F.col("currency"), 256))

    .withColumnRenamed("currency", "currency_code")

    .withColumn(

        "currency_name",

        F.coalesce(
            F.col("currency_name"),
            F.concat(F.lit("UNKNOWN — "), F.col("currency_code"))
        )

    )

    .withColumn("needs_review", F.col("currency_region").isNull())

)

In [0]:
# Write / Merge dim_currency
if DeltaTable.isDeltaTable(spark, silver_path("dim_currency")):

    target = DeltaTable.forPath(spark, silver_path("dim_currency"))

    (

        target.alias("t")

        .merge(
            dim_currency_incoming.alias("s"),
            "t.currency_code = s.currency_code"
        )

        .whenNotMatchedInsertAll()

        .execute()

    )

else:

    dim_currency_incoming.write.format("delta").mode("overwrite").save(
        silver_path("dim_currency")
    )

In [0]:
# Validate Currency Dimension
dim_currency_df = spark.read.format("delta").load(silver_path("dim_currency"))

review_count = dim_currency_df.where("needs_review = true").count()

print(
    f"dim_currency: {dim_currency_df.count()} total, "
    f"{review_count} flagged needs_review"
)

if review_count:

    display(dim_currency_df.where("needs_review = true"))

dim_currency: 18 total, 8 flagged needs_review


currency_code,currency_name,currency_region,currency_type,currency_key,needs_review
NOK,UNKNOWN — NOK,null,null,3c20cc52e71818d6efc5d8c9ca724f7db5d70f3f2890a6b97d52caf33f7d4e85,true
PLN,UNKNOWN — PLN,null,null,3ab495c54a46538a2d83751d527c3a458774adebe2510c58a05c1c0055485dab,true
ILS,UNKNOWN — ILS,null,null,4d8e4d22a600e2c7ed479322b9abbca3048a3549fa85cca2be447378f1d12303,true
KES,UNKNOWN — KES,null,null,67b9f011ed1f9689583c855603239c59d240d339d196b75434d1df0942878c90,true
HUF,UNKNOWN — HUF,null,null,951169adb6b5968fb38f0180aa37b97cddfbfed50145a21fc2494fb45fe021e9,true
CZK,UNKNOWN — CZK,null,null,9ccbfa612a826d7b19732c4f3e0d052732ea50fa59977af8b91616f8a1dceaf8,true
DKK,UNKNOWN — DKK,null,null,c529aa55a13908b476bf52b299e65a8ca0f4190406b0bf88c7fefed105317a02,true
SEK,UNKNOWN — SEK,null,null,41bc91bead554f763dfa55afcee5d512a8fb61e62e7c06c6694bca1771fa0884,true
